# 5차시 (1) BERT 한국어 감성 분류 — 영어 모델은 못 하던 걸, 우리 데이터로 가르치기

한경국립대 2026 여름특강 · 딥러닝/머신러닝 입문 (초급반)

4차시에서는 scikit-learn으로 **분류**(붓꽃)를 직접 만들었죠. 오늘은 같은 분류를 **딥러닝 모델(BERT)** 로, 그것도 **한국어 문장**으로 풉니다. 오늘의 흐름:

1. **약한 출발점** — 영어로 학습된 감성 모델을 불러와 **한국어**에 써 본다 → 거의 못 맞힌다(찍기 수준).
2. **다 같이 라벨링** — Google Sheet에 한국어 문장 + 정답(긍정/부정)을 함께 모은다.
3. **파인튜닝** — 한국어 BERT(`klue/bert-base`)를 우리 데이터로 추가 학습한다.
4. **학습 전 vs 후 비교** — 같은 테스트셋으로 영어 모델(전) ↔ 우리 모델(후) 정확도를 나란히 본다.

> 핵심 메시지: **모델이 한국어를 못 하던 게 아니라, *이 일*을 안 배웠을 뿐.** 우리가 데이터로 가르치면 잘하게 된다.

## ⚠️ 시작 전 필수: GPU 런타임 켜기

모델 학습은 **GPU** 가 있어야 빠릅니다.

1. 상단 메뉴 **[런타임] → [런타임 유형 변경]**
2. **하드웨어 가속기** 를 **T4 GPU** → **저장**
3. 다시 연결되면 아래 셀로 확인합니다. (GPU 없어도 되지만 몇 배 느립니다.)

In [ ]:
import torch

if torch.cuda.is_available():
    print("GPU 사용 가능:", torch.cuda.get_device_name(0))
else:
    print("GPU가 꺼져 있습니다. [런타임]>[런타임 유형 변경]>T4 GPU 로 설정 후 다시 실행하세요.")

### 라이브러리 설치
`transformers`(모델)·`datasets`(데이터)·`evaluate`(정확도). Colab엔 `torch`·`pandas`가 이미 있습니다.

In [ ]:
!pip install -qU transformers datasets evaluate
print("설치 완료!")

---
# Part 0 · 약한 출발점 — 영어 모델은 한국어를 모른다

먼저 **고정 테스트셋**(아래 24문장, 긍정 12 / 부정 12)을 정해 둡니다. 이 테스트셋으로 **학습 전(영어 모델)** 과 **학습 후(우리 BERT)** 를 똑같이 채점해 비교할 거예요.

In [ ]:
# 고정 테스트셋 — 학습에 쓰지 않고, before/after 비교에만 사용
KO_TEST = [
    ("정말 인생 영화였어요, 강력 추천합니다", "긍정"),
    ("배송이 생각보다 훨씬 빨라서 만족스러워요", "긍정"),
    ("음식이 따뜻하고 정성이 느껴졌습니다", "긍정"),
    ("화면이 선명하고 디자인도 고급스러워요", "긍정"),
    ("직원분이 친절해서 기분 좋게 다녀왔어요", "긍정"),
    ("가격 대비 품질이 훌륭합니다", "긍정"),
    ("오랜만에 시간 가는 줄 모르고 봤네요", "긍정"),
    ("포장이 꼼꼼해서 안심이 됐어요", "긍정"),
    ("향이 은은하고 오래가서 좋아요", "긍정"),
    ("설명이 자세해서 따라 하기 쉬웠습니다", "긍정"),
    ("기대 이상으로 깔끔하게 잘 작동해요", "긍정"),
    ("다음에도 꼭 다시 이용할 생각이에요", "긍정"),
    ("기대했는데 너무 실망스러웠어요", "부정"),
    ("배송이 일주일 넘게 걸려서 짜증났습니다", "부정"),
    ("음식이 식어서 오고 맛도 별로였어요", "부정"),
    ("화면에 자꾸 줄이 가고 불량 같아요", "부정"),
    ("직원이 불친절해서 다시는 안 갈 거예요", "부정"),
    ("가격만 비싸고 품질은 형편없네요", "부정"),
    ("중간부터 너무 지루해서 졸았습니다", "부정"),
    ("포장이 엉망이라 제품이 찌그러져 왔어요", "부정"),
    ("냄새가 이상하고 금방 사라져요", "부정"),
    ("설명서가 부실해서 한참 헤맸어요", "부정"),
    ("며칠 만에 고장 나서 환불했습니다", "부정"),
    ("두 번 다시 사고 싶지 않은 제품이에요", "부정")
]

# 영어 모델이 영어엔 잘 맞는지 확인할 소량 영어 예시
EN_DEMO = [
    ("This movie was absolutely fantastic!", "positive"),
    ("Great quality and super fast shipping.", "positive"),
    ("I love this product, highly recommend it.", "positive"),
    ("This was a complete waste of time.", "negative"),
    ("Terrible quality, very disappointed.", "negative"),
    ("I would never buy this again.", "negative")
]

print("한국어 테스트셋:", len(KO_TEST), "문장 / 영어 예시:", len(EN_DEMO))

먼저 **영어 감성 모델**을 불러와 **영어 문장**에 써 봅니다. 영어엔 잘 맞을 거예요.

In [ ]:
from transformers import pipeline

en_clf = pipeline("text-classification",
                  model="distilbert-base-uncased-finetuned-sst-2-english",
                  device=0 if torch.cuda.is_available() else -1)

for text, gold in EN_DEMO:
    p = en_clf(text)[0]
    print(f"[{gold:8s}] {text}  ->  {p['label']} ({p['score']:.2f})")

이번엔 **같은 영어 모델에 한국어**를 넣어 봅니다. (POSITIVE→긍정, NEGATIVE→부정 으로 바꿔 채점)

In [ ]:
def en_predict_ko(text):
    label = en_clf(text)[0]["label"]          # POSITIVE / NEGATIVE
    return "긍정" if label == "POSITIVE" else "부정"

correct = 0
wrong_samples = []
for text, gold in KO_TEST:
    pred = en_predict_ko(text)
    if pred == gold:
        correct += 1
    else:
        wrong_samples.append((text, gold, pred))

baseline_acc = correct / len(KO_TEST)
print(f"영어 모델의 한국어 정확도(학습 전): {baseline_acc:.0%}  ({correct}/{len(KO_TEST)})")
print("\n틀린 예시 몇 개:")
for text, gold, pred in wrong_samples[:5]:
    print(f"  정답 {gold} / 예측 {pred}  <-  {text}")

보통 **50% 안팎(찍기 수준)** 이 나옵니다. 영어 모델은 한국어 문장의 의미를 모르니까요.
→ 이제 **한국어 데이터를 우리가 모아서**, 한국어 BERT를 가르쳐 봅시다.

---
# Part A · 다 같이 데이터 라벨링하기 (Google Sheet)

좋은 분류기엔 **정답(label)이 달린 데이터** 가 필요합니다. 오늘은 **감성(긍정/부정)** 으로, 반 전체가 **Google Sheet 한 장** 을 10~20분간 함께 채웁니다. 한 사람이 몇 문장씩만 적어도 금세 **수백 개** 가 모입니다.

## A-1. 양식: `text`, `label` 두 열

새 시트를 열고(주소창에 **`sheet.new`**), 첫 줄(헤더)을 정확히 영어 소문자로 적습니다.

| text | label |
|---|---|
| 이 영화 진짜 재밌어요 | 긍정 |
| 스토리가 너무 지루했다 | 부정 |
| 배우 연기가 인상 깊었어요 | 긍정 |

**규칙**: `label` 은 `긍정`/`부정` 둘 중 하나로 **오타·공백 없이 통일**(`긍정 `(뒤 공백)·`positive` 섞이면 다른 클래스로 취급). 한 사람이 **긍정·부정 각 5문장씩** 목표.

> (응용) 감성 대신 **문의 의도**(`배송`/`환불`/`상품문의`)나 **뉴스 카테고리**로도 같은 방식이 됩니다. 처음엔 2클래스(긍/부정)가 가장 쉽고 잘 됩니다.

## A-2. 시트를 '링크 공유'로 + CSV 주소 만들기

1. 오른쪽 위 **[공유]** → 일반 액세스 **[링크가 있는 모든 사용자]** (반이 함께 채우려면 **편집자**, 수업 중에만).
2. 시트 주소 `.../d/[시트ID]/edit#gid=0` 에서 **시트ID** 와 **gid** 만 빼내 아래 셀에 채웁니다.
   (`pandas` 가 `.../export?format=csv&gid=...` 주소를 직접 읽습니다.)

In [ ]:
# TODO: 반 공용 시트의 ID 와 gid 를 채우세요 (주소 /d/ 와 /edit 사이가 시트ID)
SHEET_ID = "여기에_시트ID_붙여넣기"
GID = "0"

csv_url = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=csv&gid={GID}"
print("CSV 주소:", csv_url)

## A-3. 데이터 불러오기 (pandas)
시트가 아직이거나 에러가 나면, **다음 셀의 예비 데이터**(60문장)로 바로 진행할 수 있습니다.

In [ ]:
import pandas as pd

try:
    df = pd.read_csv(csv_url)
    df = df.dropna(subset=["text", "label"])
    df["text"] = df["text"].astype(str).str.strip()
    df["label"] = df["label"].astype(str).str.strip()
    print("시트에서 불러오기 성공! 데이터 수:", len(df))
    print(df["label"].value_counts())
    USE_SHEET = True
except Exception as e:
    print("시트를 불러오지 못했습니다 (준비 전이거나 공유 설정 확인).")
    print("오류:", e, "\n=> 아래 예비 데이터로 진행하세요.")
    USE_SHEET = False

## A-4. (예비) 시트가 없을 때 — 내장 학습 데이터 60문장
위 A-3 가 성공했다면 건너뛰어도 됩니다(시트 데이터를 덮어쓰지 않습니다).

In [ ]:
import pandas as pd

if not USE_SHEET:
    backup = [
        ("이 영화 정말 재미있었어요", "긍정"),
        ("연기가 너무 자연스럽고 좋았다", "긍정"),
        ("스토리가 탄탄해서 몰입했어요", "긍정"),
        ("음악이 분위기와 잘 어울렸다", "긍정"),
        ("기대 이상으로 만족스러웠습니다", "긍정"),
        ("다시 보고 싶은 작품이에요", "긍정"),
        ("배우들의 호흡이 환상적이었다", "긍정"),
        ("끝까지 긴장감이 넘쳤어요", "긍정"),
        ("가족과 보기 좋은 따뜻한 영화", "긍정"),
        ("여운이 오래 남는 명작이다", "긍정"),
        ("제품이 튼튼하고 마감이 깔끔해요", "긍정"),
        ("생각보다 가벼워서 들고 다니기 편해요", "긍정"),
        ("색감이 예쁘고 사진이 잘 나와요", "긍정"),
        ("가성비가 정말 뛰어난 것 같아요", "긍정"),
        ("사용법이 간단해서 바로 익혔어요", "긍정"),
        ("배터리가 오래가서 만족합니다", "긍정"),
        ("맛이 깔끔하고 자극적이지 않아요", "긍정"),
        ("양도 푸짐하고 가격도 착해요", "긍정"),
        ("매장이 깨끗하고 분위기가 좋네요", "긍정"),
        ("응대가 빠르고 정확해서 편했어요", "긍정"),
        ("택배가 다음 날 바로 도착했어요", "긍정"),
        ("선물용으로 포장이 예뻐서 좋았어요", "긍정"),
        ("기능이 다양해서 활용도가 높아요", "긍정"),
        ("소음이 거의 없어서 조용해요", "긍정"),
        ("피부에 순하고 발림성이 좋아요", "긍정"),
        ("설치가 쉬워서 금방 끝났습니다", "긍정"),
        ("화질이 또렷하고 음질도 훌륭해요", "긍정"),
        ("오래 써도 질리지 않는 디자인이에요", "긍정"),
        ("문의에 친절하게 답변해 주셨어요", "긍정"),
        ("전체적으로 아주 만족스러운 경험이었어요", "긍정"),
        ("시간이 너무 아까웠어요", "부정"),
        ("내용이 지루하고 늘어졌다", "부정"),
        ("연기가 어색해서 몰입이 안 됐다", "부정"),
        ("결말이 너무 허무했어요", "부정"),
        ("기대했는데 실망스러웠습니다", "부정"),
        ("스토리가 산만하고 엉성했다", "부정"),
        ("두 번 다시 보고 싶지 않아요", "부정"),
        ("돈이 아까운 영화였다", "부정"),
        ("전개가 뻔하고 식상했어요", "부정"),
        ("소리만 시끄럽고 알맹이가 없다", "부정"),
        ("제품이 약해서 금방 망가졌어요", "부정"),
        ("생각보다 무거워서 불편해요", "부정"),
        ("색이 사진과 완전히 달라요", "부정"),
        ("가격에 비해 품질이 너무 떨어져요", "부정"),
        ("설명이 부족해서 쓰기 어려웠어요", "부정"),
        ("배터리가 금방 닳아서 불편합니다", "부정"),
        ("맛이 너무 짜고 느끼했어요", "부정"),
        ("양이 적은데 가격은 비싸요", "부정"),
        ("매장이 지저분하고 정신없었어요", "부정"),
        ("응대가 느리고 불친절했어요", "부정"),
        ("택배가 너무 늦게 와서 답답했어요", "부정"),
        ("포장이 허술해서 깨져 있었어요", "부정"),
        ("기능이 부실해서 쓸 데가 없어요", "부정"),
        ("소음이 심해서 신경 쓰여요", "부정"),
        ("피부에 트러블이 생겨서 버렸어요", "부정"),
        ("조립이 복잡해서 한참 걸렸습니다", "부정"),
        ("화질이 흐릿하고 음질도 별로예요", "부정"),
        ("디자인이 촌스럽고 마감이 거칠어요", "부정"),
        ("문의해도 답이 없어서 답답했어요", "부정"),
        ("전체적으로 다시는 안 살 것 같아요", "부정")
    ]
    df = pd.DataFrame(backup, columns=["text", "label"])
    print("예비 데이터 사용. 데이터 수:", len(df))
    print(df["label"].value_counts())
else:
    print("시트 데이터를 사용 중이라 예비 데이터는 건너뜁니다.")

---
# Part B · 데이터 준비 (라벨→숫자, 토큰화)

모델은 글자 label(`긍정`/`부정`)을 못 읽으니 **숫자 id** 로 바꿉니다. **학습**엔 우리가 모은 데이터(`df`)를, **평가**엔 Part 0의 고정 `KO_TEST` 를 씁니다(영어 모델과 같은 잣대로 비교).

In [ ]:
label_names = sorted(df["label"].unique())
label2id = {name: i for i, name in enumerate(label_names)}
id2label = {i: name for name, i in label2id.items()}
print("클래스:", label2id)

df["label_id"] = df["label"].map(label2id)
test_df = pd.DataFrame(KO_TEST, columns=["text", "label"])
test_df["label_id"] = test_df["label"].map(label2id)
print(f"학습 {len(df)}개 / 평가(고정 테스트셋) {len(test_df)}개")

Hugging Face `datasets` 로 바꾸고, 모델과 **짝** 인 토크나이저로 문장을 토큰(숫자)으로 변환합니다.

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

MODEL_NAME = "klue/bert-base"   # 한국어로 사전학습된 BERT
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_ds = Dataset.from_pandas(df[["text","label_id"]], preserve_index=False)
test_ds  = Dataset.from_pandas(test_df[["text","label_id"]], preserve_index=False)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=64)

train_ds = train_ds.map(tokenize, batched=True).rename_column("label_id","labels")
test_ds  = test_ds.map(tokenize, batched=True).rename_column("label_id","labels")
print("토큰화 완료:", train_ds.column_names)

---
# Part C · BERT 파인튜닝 (모델 학습)

사전학습된 BERT 위에 **우리 클래스 수에 맞는 분류층** 을 얹어 학습합니다. 처음 실행 시 모델(수백 MB)을 내려받느라 잠깐 걸립니다.

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(label_names), id2label=id2label, label2id=label2id)
print("모델 준비 완료. 클래스 수:", len(label_names))

**정확도** 를 지표로, `Trainer` 로 학습합니다. 데이터가 적으면 GPU에서 1~3분이면 끝납니다.

In [ ]:
import numpy as np, evaluate
from transformers import TrainingArguments, Trainer

accuracy = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return accuracy.compute(predictions=np.argmax(logits, axis=-1), references=labels)

args = TrainingArguments(
    output_dir="bert_classifier", num_train_epochs=5,
    per_device_train_batch_size=8, per_device_eval_batch_size=8,
    learning_rate=2e-5, eval_strategy="epoch", logging_steps=10, report_to="none")

trainer = Trainer(model=model, args=args, train_dataset=train_ds,
                  eval_dataset=test_ds, compute_metrics=compute_metrics)
trainer.train()

---
# Part D · 학습 전 vs 학습 후 — 같은 테스트셋으로 비교

드디어 비교입니다. 같은 `KO_TEST` 에서, **영어 모델(학습 전)** 과 **우리가 가르친 BERT(학습 후)** 의 정확도를 나란히 봅니다.

In [ ]:
finetuned = trainer.evaluate()
finetuned_acc = finetuned["eval_accuracy"]

print("=== 같은 한국어 테스트셋 24문장 채점 ===")
print(f"학습 전 (영어 모델)     : {baseline_acc:.0%}")
print(f"학습 후 (우리 한국어 BERT): {finetuned_acc:.0%}")
print(f"향상폭                  : {finetuned_acc - baseline_acc:+.0%}")

학습 전엔 틀렸던 문장들이, 학습 후엔 어떻게 바뀌었는지 직접 봅시다.

In [ ]:
from transformers import pipeline
ko_clf = pipeline("text-classification", model=model, tokenizer=tokenizer,
                  device=0 if torch.cuda.is_available() else -1)

print(f"{'정답':4} | {'학습 전(영어)':12} | {'학습 후(BERT)':12} | 문장")
for text, gold in KO_TEST[:8]:
    before = en_predict_ko(text)
    after  = ko_clf(text)[0]["label"]
    print(f"{gold:4} | {before:12} | {after:12} | {text}")

---
# Part E · 새 문장으로 예측해 보기
학습한 모델로 처음 보는 문장을 분류해 봅니다.

In [ ]:
test_sentences = ["이건 정말 인생 영화예요", "보다가 졸았습니다", "생각보다 괜찮았어요"]
for s in test_sentences:
    out = ko_clf(s)[0]
    print(f"'{s}' -> {out['label']} (확신도 {out['score']:.2f})")

**연습.** 아래에 여러분의 문장을 넣어 보세요. 틀리는 문장이 있다면, 그런 문장을 시트에 더 모아 다시 학습하면 좋아집니다("데이터가 곧 성능").

In [ ]:
my_sentences = [
    "",  # 예: "이 가격에 이 정도면 훌륭하죠"
]
for s in my_sentences:
    if s.strip():
        out = ko_clf(s)[0]
        print(f"'{s}' -> {out['label']} (확신도 {out['score']:.2f})")

---
## (심화·선택) 분류기를 챗봇의 '라우터' 로 쓰기

방금 만든 감성 분류기는 다음(에이전트) 노트북의 **라우팅** 에 그대로 쓸 수 있습니다. 예를 들어 고객센터 챗봇이 들어온 말의 감성을 분류해 **부정 → 사과·상담 연결**, **긍정 → 감사 인사** 로 보내는 식입니다. (의도 `배송`/`환불` 데이터로 학습하면 의도 라우터가 됩니다.)

In [ ]:
def route_by_sentiment(message):
    label = ko_clf(message)[0]["label"]
    if label == "부정":
        return "[상담 연결] 불편을 드려 죄송합니다. 어떤 점이 문제였는지 도와드릴게요."
    else:
        return "[감사] 좋게 봐 주셔서 감사합니다! 또 찾아 주세요."

print(route_by_sentiment("배송도 빠르고 너무 만족해요"))
print(route_by_sentiment("환불하고 싶은데 연락이 안 돼요"))

---
## 마무리

- **학습 전**: 영어 모델은 한국어 감성을 거의 못 맞혔습니다(≈찍기).
- **우리가 한 일**: 한국어 문장에 긍정/부정을 **직접 라벨링** → 한국어 BERT(`klue/bert-base`)를 **파인튜닝**.
- **학습 후**: 같은 테스트셋에서 정확도가 크게 올랐습니다. **데이터로 가르치니 잘하게 됐다.**
- 데이터가 많고 라벨이 깔끔할수록 더 좋아집니다("데이터가 곧 성능"). 이 분류기는 다음 노트북의 **라우터** 로도 재활용됩니다.

> 모델 저장: `trainer.save_model("my_bert")` → 나중에 `pipeline(model="my_bert")` 로 다시 사용.